# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umemon254/Fly-rank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# Setup
from google.colab import userdata
import os
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("DROP SECRET IF EXISTS hf;")
con.execute(f"""
CREATE SECRET hf (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
);
""")

print("Setup complete")


Setup complete


In [2]:
# Build one row per content item + client for March.
# April is kept separate so future information does not enter the features.

march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
april_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

march = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS clicks_march,
    SUM(gsc_impressions) AS impressions_march,
    SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_march,
    SUM(gsc_avg_position * gsc_impressions)
        / NULLIF(SUM(gsc_impressions), 0) AS position_march,
    SUM(sessions_organic) AS organic_sessions_march,
    SUM(sessions_ai) AS ai_sessions_march,
    SUM(sessions_paid) AS paid_sessions_march,
    SUM(sessions_referral) AS referral_sessions_march
FROM read_parquet('{march_path}')
WHERE client_has_gsc IS TRUE
  AND gsc_impressions > 0
GROUP BY client_hash_id, content_hash_id
""").df()

april = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS clicks_april,
    SUM(gsc_impressions) AS impressions_april
FROM read_parquet('{april_path}')
WHERE client_has_gsc IS TRUE
  AND gsc_impressions > 0
GROUP BY client_hash_id, content_hash_id
""").df()

df = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Define the future outcome only after the March feature frame is complete.
# A positive label means April clicks were at least 20% lower than March clicks.
df = df[df["clicks_march"] > 0].copy()
df["is_declining"] = (
    df["clicks_april"] <= 0.80 * df["clicks_march"]
).astype(int)

print("Feature rows:", len(df))
print("Declining rate:", round(df["is_declining"].mean(), 3))
df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 68040
Declining rate: 0.61


,client_hash_id,content_hash_id,clicks_march,impressions_march,ctr_march,position_march,organic_sessions_march,ai_sessions_march,paid_sessions_march,referral_sessions_march,clicks_april,impressions_april,is_declining
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,22.0,10849.0,0.002028,8.049866,NaN,NaN,NaN,NaN,23.0,16121.0,0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,1.0,705.0,0.001418,5.863830,NaN,NaN,NaN,NaN,0.0,213.0,1
5,client_62f4a7e64f5e0096,content_df22bda1218f13ff,1.0,2099.0,0.000476,2.632206,NaN,NaN,NaN,NaN,4.0,1472.0,0
7,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,25.0,3535.0,0.007072,3.151344,NaN,NaN,NaN,NaN,34.0,10500.0,0
8,client_62f4a7e64f5e0096,content_d7568011c4325a33,3.0,1558.0,0.001926,5.247112,NaN,NaN,NaN,NaN,1.0,746.0,1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# Client-level split
RANDOM_STATE = 42

clients = df["client_hash_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
clients = rng.permutation(clients)

test_client_count = max(1, int(round(len(clients) * 0.20)))
test_clients = set(clients[:test_client_count])

test_mask = df["client_hash_id"].isin(test_clients)

train_df = df.loc[~test_mask].copy()
test_df = df.loc[test_mask].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())
print("Train positive rate:", round(train_df["is_declining"].mean(), 3))
print("Test positive rate:", round(test_df["is_declining"].mean(), 3))


Train rows: 60541
Test rows: 7499
Train clients: 34
Test clients: 8
Train positive rate: 0.595
Test positive rate: 0.732


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
FEATURES = [
    "clicks_march",
    "impressions_march",
    "ctr_march",
    "position_march",
    "organic_sessions_march",
    "ai_sessions_march",
    "paid_sessions_march",
    "referral_sessions_march",
]

X_train = train_df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = test_df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df["is_declining"].astype(int)
y_test = test_df["is_declining"].astype(int)

def precision_at_k(y_true, scores, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean()) if k else 0.0

# Baseline thresholds are learned from TRAIN only.
ctr_threshold = train_df["ctr_march"].quantile(0.25)
position_threshold = train_df["position_march"].quantile(0.75)

baseline_scores = (
    (test_df["ctr_march"] <= ctr_threshold).astype(int)
    + (test_df["position_march"] >= position_threshold).astype(int)
).to_numpy()

# Models
logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

logistic.fit(X_train, y_train)
forest.fit(X_train, y_train)

logistic_scores = logistic.predict_proba(X_test)[:, 1]
forest_scores = forest.predict_proba(X_test)[:, 1]

def metrics(name, scores):
    preds = (scores >= 0.5).astype(int)
    return {
        "method": name,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
        "precision_at_20": precision_at_k(y_test.to_numpy(), scores, 20),
        "precision_at_50": precision_at_k(y_test.to_numpy(), scores, 50),
    }

# Baseline score is a rank score, so evaluate it with ranking metrics.
results = pd.DataFrame([
    metrics("Baseline rule", baseline_scores),
    metrics("Logistic Regression", logistic_scores),
    metrics("Random Forest", forest_scores),
])

results.round(3)


,method,precision,recall,f1,roc_auc,average_precision,precision_at_20,precision_at_50
0,Baseline rule,0.763,0.407,0.531,0.526,0.740,0.60,0.66
1,Logistic Regression,0.726,0.475,0.574,0.477,0.726,0.85,0.80
2,Random Forest,0.734,0.399,0.517,0.512,0.741,0.75,0.84


In [5]:
# Feature interpretation: permutation importance on the held-out clients.
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    logistic,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

importance.round(4)


,feature,importance
3,position_march,0.0165
5,ai_sessions_march,0.0004
6,paid_sessions_march,-0.0001
7,referral_sessions_march,-0.0002
1,impressions_march,-0.0003
0,clicks_march,-0.0010
4,organic_sessions_march,-0.0045
2,ctr_march,-0.0082


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# Error analysis for the Logistic Regression model
error_df = test_df[
    ["client_hash_id", "content_hash_id", "clicks_march",
     "impressions_march", "ctr_march", "position_march", "is_declining"]
].copy()

error_df["model_score"] = logistic_scores
error_df["prediction"] = (error_df["model_score"] >= 0.5).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df["prediction"] == 1) & (error_df["is_declining"] == 0),
        (error_df["prediction"] == 0) & (error_df["is_declining"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct"
)

print("False positives:", (error_df["error_type"] == "false_positive").sum())
print("False negatives:", (error_df["error_type"] == "false_negative").sum())

print("\nExamples of false positives:")
display(
    error_df[error_df["error_type"] == "false_positive"]
    .sort_values("model_score", ascending=False)
    .head(3)
)

print("\nExamples of false negatives:")
display(
    error_df[error_df["error_type"] == "false_negative"]
    .sort_values("model_score", ascending=False)
    .head(3)
)


False positives: 984
False negatives: 2884

Examples of false positives:


,client_hash_id,content_hash_id,clicks_march,impressions_march,ctr_march,position_march,is_declining,model_score,prediction,error_type
158407,client_1a730cb2640a1abf,content_52d26fc5cffe2f17,1.0,1.0,1.000000,0.0,0,0.999998,1,false_positive
155710,client_1a730cb2640a1abf,content_69a9e969984e57fe,1.0,5.0,0.200000,5.4,0,0.931791,1,false_positive
158415,client_1a730cb2640a1abf,content_4d0628bcd2609a67,1.0,6.0,0.166667,5.5,0,0.898299,1,false_positive



Examples of false negatives:


,client_hash_id,content_hash_id,clicks_march,impressions_march,ctr_march,position_march,is_declining,model_score,prediction,error_type
10724,client_fef1a8f436438636,content_0dcc0bdaa94896a6,2.0,1046.0,0.001912,7.877629,1,0.499997,0,false_negative
27157,client_fef1a8f436438636,content_00a6e42a2bf75f43,1.0,723.0,0.001383,15.923928,1,0.499993,0,false_negative
27521,client_fef1a8f436438636,content_9353e1cd57b92473,2.0,1702.0,0.001175,13.347826,1,0.499957,0,false_negative


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.